In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# ============================
# 1. Load Sydney Data
# ============================

df_sydney = pd.read_csv('Sydney_SA1_variables_for_trips.csv')# Replace with the variables of the target city

variables = [
    'Weighted Population',
    '%of commercial landuse',
    'station', 
    'Weekly household income'
]

sydney_data = df_sydney[['SA1_CODE_2021'] + variables].copy()
sydney_data.dropna(inplace=True)

# ============================
# 2. Standardize Variables
# ============================

scaler = StandardScaler()
X_scaled = scaler.fit_transform(sydney_data[variables])

# Convert back to DataFrame
X_scaled = pd.DataFrame(
    X_scaled,
    columns=[v + '_scaled' for v in variables],
    index=sydney_data.index
)

sydney_data = pd.concat([sydney_data, X_scaled], axis=1)

# ============================
# 3. Apply Seattle Coefficients
# ============================

beta_0 = 6.411
beta_wp = 0.182
beta_comm = 0.365
beta_station = 0.043
beta_income = -0.102

linear_part = (
    beta_0
    + beta_wp * sydney_data['Weighted Population_scaled']
    + beta_comm * sydney_data['%of commercial landuse_scaled']
    + beta_station * sydney_data['station_scaled']
    + beta_income * sydney_data['Weekly household income_scaled']
)

# Negative Binomial with log link
sydney_data['Predicted_Weighted_Trips'] = np.exp(linear_part)

# ============================
# 4. Save Results
# ============================

predicted_sydney = sydney_data[['SA1_CODE_2021', 'Predicted_Weighted_Trips']]

print(predicted_sydney.head())
print("\n✅ Predictions completed successfully using StandardScaler!")

   SA1_CODE_2021  Predicted_Weighted_Trips
0    10102100707                581.044737
1    10102100710                671.135160
2    10104102401                527.958745
3    10104102402                519.075110
4    10104102412                520.279648

✅ Predictions completed successfully using StandardScaler!


In [4]:
df_sydney .columns

Index(['Unnamed: 0', 'SA1_CODE_2021', 'Weighted Population',
       'Weighted vehicles', 'Household median age', 'Weekly household income',
       'AREASQKM21_x', 'station', 'POIs', '%of commercial landuse',
       '%of education landuse', '%of hospital landuse',
       '%of industrial landuse', '%of other landuse', '%of parkland landuse',
       '%of primary production landuse', '%of residential landuse',
       '%of transport landuse', '%of water landuse', 'Vehicle/Population',
       'Population density', 'Station density', 'POI density',
       'Vehicle density', 'Total Students', 'total employed',
       'employed density', 'Student density'],
      dtype='object')

In [6]:
predicted_sydney.to_csv(
    'Sydney_Predicted_Weighted_Trips_NegBin_Seattle.csv',
    index=False
)